# Real-world data coding for neuroscientists (ReCoN)

### MSc in Translational Neuroscience,

### Department of Brain Sciences, Faculty of Medicine,

### Imperial College London

### Contributors: Anastasia Ilina, Jared Maina, Cecilia Rodriguez, Marirena Bafaloukou, Katarzyna Marta Zoltowska, Rishideep Chatterjee, Sahar Rahbar, Cynthia Sandor

### Autumn 2026

---

## Tutorial 2 – Tabular data with pandas on Electronic Healthcare Records (EHR)

### **Part 1: Data visualisation. Data cleaning and preprocessing with pandas**


---

## Learning outcomes

**Overall aim**: prepare real-world clinical data for statistical analysis.

**You will learn to:**
- Visualise distributions, trends and relationships in the data.
- Flag outliers and data-entry errors, and decide what to do with them.
- Find and resolve duplicate records.
- Describe, diagnose and handle missing data.

The notebook has five sections, each ending in a ✅ **checkpoint**. If you can tick everything off, move on; if not, revisit that section first.

| Section | Key tools |
| --- | --- |
| 1. Categorical data | `value_counts`, `pd.cut`, `pd.crosstab`, `.plot` |
| 2. Continuous data | `describe()` and summary measures, histograms, `np.log1p`, seaborn box / violin / scatter plots |
| 3. Outliers | `.quantile`, `.between`, `.clip`, boolean masks |
| 4. Duplicates | `.duplicated`, `.drop_duplicates`, `.groupby().agg` |
| 5. Missing data | `.isna`, `.dropna`, `.fillna`, `IterativeImputer` |

---
## 📚 Supporting materials

> - [🐼 Pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html) and [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
> - [📊 Matplotlib cheat sheets](https://matplotlib.org/cheatsheets/cheatsheets.pdf) and [beginner handout](https://matplotlib.org/cheatsheets/handout-beginner.pdf)
> - [📉 Seaborn documentation](https://seaborn.pydata.org/index.html)
> - [📶 Data Analysis by Mike Nguyen](https://bookdown.org/mike/data_analysis/): statistics and machine learning reference

---
## Setup

Run the two cells below. The first imports the libraries; the second builds a synthetic Parkinson's disease EHR dataset with deliberately planted errors, duplicates and missing values. You don't need to read the generation code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
# === RUN THIS CELL TO (RE)CREATE THE SYNTHETIC DATASET ===
# To rebuild from scratch: run `del df`, then run this cell again.
expected_cols = {"LEDD","sex","years_since_dx","age","updrs_iii","patient_id","ethnicity","BMI"}

# Only (re)create if df isn't already present with expected columns
if not (isinstance(globals().get("df"), pd.DataFrame) and expected_cols.issubset(df.columns)):
    rng = np.random.default_rng(0)
    n = 5000

    # ---- Core table ----
    df = pd.DataFrame({
        "patient_id": [f"P{i:04d}" for i in range(n)],
        "sex": rng.choice(["Male","Female"], size=n, p=[0.55, 0.45]),
        "years_since_dx": rng.uniform(0, 12, size=n).round(2),
        "age": rng.normal(62, 8, size=n).round(1),
        "ethnicity": rng.choice(["White", "Black", "Asian", "Other", ""],
                                size=n, p=[0.70, 0.10, 0.10, 0.05, 0.05]),
        "BMI": rng.normal(27, 4, size=n).round(1),
    })

    # ---- LEDD: base + exponential noise (skewed) ----
    df["LEDD"] = (250 + 50*df["years_since_dx"] + rng.exponential(500, size=n)).clip(0).round(1)

    # ---- UPDRS III: realistic 0–132 with progression + heterogeneity ----
    # Patient-level random intercepts (baseline severity) and slopes (annual change)
    unique_ids = df["patient_id"].unique()
    intercepts = pd.Series(np.clip(rng.normal(20, 6, size=len(unique_ids)), 0, 60), index=unique_ids)
    slopes     = pd.Series(np.clip(rng.normal(2.2, 0.9, size=len(unique_ids)), 0, 5), index=unique_ids)

    alpha = df["patient_id"].map(intercepts).to_numpy()
    beta  = df["patient_id"].map(slopes).to_numpy()
    years = df["years_since_dx"].astype(float).to_numpy()
    eps   = rng.normal(0, 5, size=len(df))  # visit-level noise

    updrs = np.clip(alpha + beta*years + eps, 0, 132)
    df["updrs_iii"] = np.round(updrs, 1)

    # Inject a few impossible outliers (>132) for teaching Winsorisation/capping
    n_out = max(20, int(0.004 * len(df)))  # ~0.4% or at least 20 rows
    idx_out = rng.choice(df.index, size=n_out, replace=False)
    df.loc[idx_out, "updrs_iii"] = np.round(rng.uniform(140, 220, size=n_out), 1)

    # ---- Intentional data-entry errors (for cleaning exercises) ----
    # BMI: ×10 typos and negatives
    df.loc[rng.choice(df.index, size=5, replace=False), "BMI"] *= 10
    df.loc[rng.choice(df.index, size=5, replace=False), "BMI"] *= -1
    # Age: negatives and unrealistic high values
    df.loc[rng.choice(df.index, size=5, replace=False), "age"] *= -1
    df.loc[rng.choice(df.index, size=5, replace=False), "age"] += 108

    # ---- Duplicate visits for some patients (longitudinal flavor) ----
    ###reps = df.sample(200, replace=True, random_state=0).copy()
    ##reps["patient_id"] = reps["patient_id"].sample(frac=1.0, random_state=1).values
    #df = pd.concat([df, reps], ignore_index=True)

    # make LEDD and UPDRS higher on average for men 
    df.loc[df["sex"]=="Male", "LEDD"] *= rng.uniform(1.05, 1.15, size=(df["sex"]=="Male").sum())
    df.loc[df["sex"]=="Male", "updrs_iii"] *= rng.uniform(1.05, 1.15, size=(df["sex"]=="Male").sum())

    # ---- MCAR missingness on selected columns ----
    for col, p in [("LEDD", 0.05), ("updrs_iii", 0.10), ("sex", 0.01), ("age", 0.02), ("ethnicity", 0.05)]:
        df.loc[rng.choice(df.index, size=int(p*len(df)), replace=False), col] = np.nan

    # ---- MNAR: LEDD more likely missing when UPDRS is high ----
    high_updrs = df["updrs_iii"] > 45
    # Safeguard: drop NaNs before sampling indices
    high_idx = df.index[high_updrs.fillna(False)]
    if len(high_idx) > 0:
        df.loc[rng.choice(high_idx, size=int(0.3*len(high_idx)), replace=False), "LEDD"] = np.nan

    df.reset_index(drop=True, inplace=True)

    # ---- Summary printout ----
    print(f"Created synthetic dataset with {len(df)} rows")
    print(f"Injected UPDRS>132 outliers: {(df['updrs_iii'] > 132).sum()}")
    print("Missingness (%):")
    print((df[["LEDD","updrs_iii","sex","age","ethnicity","BMI"]].isna().mean()*100).round(1))

    # ---- Duplicates in three buckets (exact, same-visit, legit repeat visits) ----
    # Counts: adjust as you like
    n_exact = 40          # exact row duplicates
    n_key   = 160         # same patient_id & years_since_dx (not exact)
    n_repeat= 300         # legit repeated visits (different years_since_dx)

    df_list = [df]

    # --- 1) Exact row duplicates: true copy-paste of entire rows ---
    idx_exact = rng.choice(df.index, size=n_exact, replace=False)
    exact_dups = df.loc[idx_exact].copy()
    df_list.append(exact_dups)

    # --- 2) Key duplicates: same patient_id & same years_since_dx but slight entry drift ---
    # pick base rows
    idx_key = rng.choice(df.index.difference(idx_exact), size=n_key, replace=False)
    key_dups = df.loc[idx_key].copy()

    # Introduce small, realistic data-entry drift to numeric fields (but keep ID + time the same)
    def small_perturb(x, scale):
        return np.round(x + rng.normal(0, scale, size=len(x)), 1)

    if "LEDD" in key_dups:
        key_dups["LEDD"] = small_perturb(key_dups["LEDD"], scale=15)     # ±15 mg
    if "updrs_iii" in key_dups:
        key_dups["updrs_iii"] = np.clip(small_perturb(key_dups["updrs_iii"], scale=1.5), 0, 220)
    if "BMI" in key_dups:
        key_dups["BMI"] = np.round(key_dups["BMI"] + rng.normal(0, 0.3, size=len(key_dups)), 1)

    # keep patient_id and years_since_dx unchanged (this makes them key duplicates)
    df_list.append(key_dups)

    # --- 3) Legit repeats: same patient, NEW years_since_dx (and recomputed fields) ---
    # choose patients to get an extra visit
    rep_patients = rng.choice(df["patient_id"].unique(), size=n_repeat, replace=False)

    # For each chosen patient, create a new visit at a later (or earlier) year_since_dx
    reps = df[df["patient_id"].isin(rep_patients)].copy()

    # Propose a new time that differs from the current one by ~0.5–2.0 years
    delta = rng.uniform(0.5, 2.0, size=len(reps))
    sign  = rng.choice([1, -1], size=len(reps), p=[0.8, 0.2])  # mostly later visits
    new_years = np.clip(reps["years_since_dx"].astype(float).to_numpy() + sign*delta, 0, 12)
    reps["years_since_dx"] = np.round(new_years, 2)

    # Age moves with time (if you want): comment out if not desired
    if "age" in reps:
        reps["age"] = np.round(reps["age"].astype(float) + sign*delta, 1)

    # Recompute LEDD and UPDRS for the *new* visit using the same subject-specific intercept/slope
    pid = reps["patient_id"]
    alpha_rep = pid.map(intercepts).to_numpy()
    beta_rep  = pid.map(slopes).to_numpy()
    years_rep = reps["years_since_dx"].astype(float).to_numpy()

    # fresh noise terms
    eps_rep = rng.normal(0, 5, size=len(reps))
    updrs_rep = np.clip(alpha_rep + beta_rep*years_rep + eps_rep, 0, 132)
    reps["updrs_iii"] = np.round(updrs_rep, 1)

    # recompute LEDD with new years_since_dx
    led_noise = rng.exponential(500, size=len(reps))
    reps["LEDD"] = (250 + 50*reps["years_since_dx"].astype(float) + led_noise).clip(0).round(1)

    # Append legit repeats
    df = pd.concat(df_list + [reps], ignore_index=True)

    # ---- Quick bookkeeping prints ----
    print("\nInjected duplicates summary:")
    print(f"  Exact duplicates added:         {len(exact_dups)}")
    print(f"  Key-duplicates added:           {len(key_dups)}")
    print(f"  Legit repeat visits added:      {len(reps)}")

    # (OPTIONAL) Sanity counts right now (pre-missingness/multipliers)
    exact_count = df.duplicated(keep=False).sum()
    key_count   = df.duplicated(subset=['patient_id','years_since_dx'], keep=False).sum()
    print(f"  Current exact-dup rows (counted individually): {exact_count}")
    print(f"  Current key-dup rows (same patient+time):      {key_count}")

### A first look

Each row is one clinic visit, and `patient_id` identifies the patient, so one patient can appear in several rows.

In [ ]:
display(df.head())
print(f"{len(df)} visits from {df['patient_id'].nunique()} patients")

---
## 1. Visualising categorical data

### Counts and proportions

`value_counts()` tallies each category, and pandas can plot the result directly. EHR systems often record a missing label as `NaN` in some rows and as an empty string in others, so we merge both into an explicit "Unknown" category. Hiding the unknowns would distort the percentages.

In [ ]:
ethnicity = df["ethnicity"].replace("", np.nan).fillna("Unknown")   # treat empty strings as missing, then label all missing values "Unknown"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))   # one figure, two panels side by side: axes[0] (left) and axes[1] (right)
ax = ethnicity.value_counts().plot.bar(ax=axes[0], edgecolor="black", rot=20, title="Ethnicity: counts")
ax.bar_label(ax.containers[0])   # write each bar's value on top of it
# normalize=True gives proportions (0–1); × 100 turns them into percentages
ax = (ethnicity.value_counts(normalize=True) * 100).plot.bar(ax=axes[1], edgecolor="black", rot=20, title="Ethnicity: percent")
ax.bar_label(ax.containers[0], fmt="%.1f%%")
axes[0].set_ylabel("Count")
axes[1].set_ylabel("Percent (%)")
plt.tight_layout()
plt.show()

> 💡 seaborn's `sns.countplot(x=ethnicity)` draws the same chart. Use whichever you find more readable.

### From continuous to categorical

`pd.cut` sorts numeric values into labelled bins, such as the WHO BMI categories. `.reindex(labels)` keeps the bars in clinical order rather than sorting them by frequency.

In [ ]:
bmi_labels = ["Underweight", "Normal", "Overweight", "Obesity I", "Obesity II+"]
# right=False: each bin includes its left edge, so 25.0 counts as "Overweight", not "Normal"
bmi_category = pd.cut(df["BMI"], bins=[0, 18.5, 25, 30, 35, 100], labels=bmi_labels, right=False)

ax = bmi_category.value_counts().reindex(bmi_labels).plot.bar(edgecolor="black", rot=20, title="BMI categories")
ax.bar_label(ax.containers[0])
plt.show()

### Two categorical variables: `pd.crosstab`

`pd.crosstab` counts every combination of two categories. The resulting table plots directly as:
- **grouped bars**, to compare counts;
- **stacked bars**, to show totals *and* how each total is made up;
- **100% stacked bars**, to compare how each group is made up, regardless of size.

In [ ]:
sex = df["sex"].fillna("Unknown")

ct = pd.crosstab(ethnicity, sex)
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]   # order rows by total
ct_pct = pd.crosstab(ethnicity, sex, normalize="index").loc[ct.index] * 100   # normalize="index": each row sums to 100%; .loc keeps the row order of ct
display(ct)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ct.plot.bar(ax=axes[0], edgecolor="black", rot=20, title="Grouped")
ct.plot.bar(ax=axes[1], stacked=True, edgecolor="black", rot=20, title="Stacked")
ct_pct.plot.bar(ax=axes[2], stacked=True, edgecolor="black", rot=20, title="100% stacked", legend=False)
axes[0].set_ylabel("Count")
axes[2].set_ylabel("Percent (%)")
plt.tight_layout()
plt.show()

> 💡 Merge very small categories into "Other" for readability, and set the order explicitly when categories have a natural one (e.g. disease stage).

#### <span style="color:red"> 🍎 Exercise 1: your turn! </span>

<span style="color:red">

* The UPDRS III motor scale classifies scores of 0–32 as **Mild**, 33–58 as **Moderate** and 59 or more as **Severe**. Create a column `UPDRS_Severity` with `pd.cut` and plot how many visits fall into each group.
* Compare the severity groups by sex using a grouped bar chart.
* Show how severity is distributed within each ethnicity using a 100% stacked bar chart.

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 1

Before moving on, check that you can:
- count and plot categories with `value_counts()`, including an explicit "Unknown" group;
- bin a continuous variable with `pd.cut`;
- cross-tabulate two categorical variables, and choose between grouped, stacked and 100% stacked bars.

---
## 2. Visualising continuous data

### Summary measures

Before plotting a numeric column, summarise it with a few numbers. The **mean, median and mode** describe the centre of a distribution; the **standard deviation, variance, range and quartiles** describe its spread; and **skewness and kurtosis** describe its shape. pandas has a method for each:

| Statistic | Code | Meaning |
| --- | --- | --- |
| **Mean** (average) | `df["updrs_iii"].mean()` | The *centre* of the data: total divided by the number of observations. |
| **Median** | `df["updrs_iii"].median()` | The *middle* value when sorted; less sensitive to outliers. |
| **Mode** | `df["updrs_iii"].mode()` | The *most frequent* value in the column. |
| **Standard deviation (SD)** | `df["updrs_iii"].std()` | Typical distance of values from the mean: a measure of *spread*. |
| **Variance** | `df["updrs_iii"].var()` | SD squared; another measure of spread. |
| **Minimum / Maximum** | `df["updrs_iii"].min()`, `.max()` | Smallest and largest values. |
| **Count (non-missing)** | `df["updrs_iii"].count()` | Number of non-NaN values. |
| **Sum** | `df["updrs_iii"].sum()` | Total of all values. |
| **Quantiles** | `df["updrs_iii"].quantile([0.25, 0.5, 0.75])` | 25th, 50th (median) and 75th percentiles. |
| **Skewness** | `df["updrs_iii"].skew()` | Whether the distribution is lopsided (long tail to the right or left). |
| **Kurtosis** | `df["updrs_iii"].kurt()` | How "peaked" or "flat" the distribution is. |

![Mean, median and mode in a symmetrical and a skewed distribution](images/mean_median_mode.jpg)

In a symmetrical distribution the mean, median and mode coincide. In a skewed distribution they separate: the long tail pulls the mean towards it, while the median stays closer to the bulk of the data.

`describe()` calculates the most common measures in one go:

In [ ]:
df[["updrs_iii", "LEDD"]].describe()   # count, mean, std (SD), min, 25%, 50% (median), 75% and max

#### <span style="color:red"> 🍎 Exercise 2: your turn! </span>

<span style="color:red">

* Calculate the mean, median, mode, standard deviation, variance, min, max, count, sum, 25th, 50th and 75th percentiles, skewness and kurtosis of the `updrs_iii` column. *Hint: `.mode()` returns a Series, so take its first value with `[0]`.*
* Which of these values could you have read straight from the `describe()` table?
* Does anything in your results look clinically impossible? (We come back to this in Section 3.)

</span>

In [ ]:
# Write your code here:

#### <span style="color:red"> 🍎 Bonus question! </span>

![](images/IMG_2876.JPG)

* Which of these cats has the **higher standard deviation**? Why?
* Which one has the **higher kurtosis**? Why?

Answer:

### Histograms

A histogram shows the shape of a distribution. The number of bins matters: too few bins hide the structure, while too many turn it into noise. To save a figure, call `plt.savefig()` before `plt.show()`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["age"].plot.hist(ax=axes[0], bins=30, edgecolor="black", title="Age (years)")
df["LEDD"].plot.hist(ax=axes[1], bins=30, edgecolor="black", title="LEDD (mg/day)")
plt.tight_layout()
plt.savefig("age_ledd_histograms.png", dpi=300)   # saves next to the notebook; dpi=300 is print quality
plt.show()

Age is roughly symmetric (ignore the odd values for now; we come back to them in Section 3). LEDD has a long **right tail**: a few patients take much higher doses than everyone else. Skewed distributions like this are common in clinical data.

#### <span style="color:red"> 🍎 Exercise 3: your turn! </span>

<span style="color:red">

* Using the figure in *Summary measures* above, predict the order in which the **mean, median and mode** of LEDD will fall.
* Plot all three as vertical lines (`plt.axvline`) on the LEDD histogram to check your answer, and show their values in the legend.
* Redraw the histogram with 5, 10, 20, 50 and 100 bins. How does the shape change?

</span>

In [ ]:
# Write your code here:

### <span style="color:red">If you've done things correctly, you should see something like this:</span>

![](images/IMG_2882.jpg)

### <span style="color:red">The value of the mode may be surprising. Why?</span>

### Log scales for skewed data

![](images/IMG_2871.WEBP)

A log transform compresses large values and stretches small ones, so a long tail no longer dominates the plot. There are two ways to use it:
1. **Log-scale the axis**: the data stay the same and only the axis is rescaled.
2. **Log-transform the variable**: `np.log1p(x)` computes log(x + 1), which also handles zeros.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# logy=True rescales the y-axis only; np.log1p (below) transforms the data itself
df["LEDD"].plot.hist(ax=axes[0], bins=50, edgecolor="black", logy=True, title="LEDD, log-scaled y-axis")
np.log1p(df["LEDD"]).plot.hist(ax=axes[1], bins=30, edgecolor="black", title="log(LEDD + 1)")
plt.tight_layout()
plt.show()

### Comparing groups: box and violin plots

A **box plot** summarises a distribution without assuming its shape:

| Component | Meaning |
| --- | --- |
| Box | Interquartile range (IQR): the middle 50% of values, from Q1 (25th percentile) to Q3 (75th) |
| Line inside the box | Median |
| Whiskers | Reach the most extreme values within Q1 − 1.5 × IQR and Q3 + 1.5 × IQR |
| Dots | Values beyond the whiskers: *potential* outliers |

A **violin plot** also shows the distribution's shape (a smoothed density), which reveals skew or multiple peaks that a box plot hides.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
sns.boxplot(data=df, x="sex", y="LEDD", ax=axes[0])
# cut=0 stops the violin extending beyond the smallest and largest observed values
sns.violinplot(data=df, x="sex", y="LEDD", inner="quartile", cut=0, ax=axes[1])
axes[0].set_title("Box plot")
axes[1].set_title("Violin plot (dashed lines = quartiles)")
plt.tight_layout()
plt.show()

Summary plots can hide how many patients sit behind them. Overlaying a **strip plot** shows each individual; with thousands of rows, plot a random sample (`random_state` makes the sample reproducible). `hue` splits each group by a second variable.

In [ ]:
df_sample = df.sample(800, random_state=1).assign(   # random rows; random_state makes the same sample every run
    age_group=lambda d: pd.cut(d["age"], bins=[0, 55, 120], labels=["≤55", "56+"])   # .assign adds a column; d is the sampled DataFrame
)

plt.figure(figsize=(7, 4))
sns.boxplot(data=df_sample, x="sex", y="LEDD", hue="age_group", showfliers=False)
# dodge=True lines the points up with their boxes; legend=False avoids a second legend
sns.stripplot(data=df_sample, x="sex", y="LEDD", hue="age_group", dodge=True,
              palette=["0.25", "0.25"], size=3, alpha=0.5, legend=False)
plt.title("LEDD by sex and age group")
plt.show()

### Relationships between two continuous variables: scatter plots

Each point is one visit. Use `alpha` (transparency) to reduce overplotting, and `hue` to colour points by group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# alpha=0.3 makes points semi-transparent, so crowded areas look darker
sns.scatterplot(data=df, x="years_since_dx", y="updrs_iii", alpha=0.3, edgecolor=None, ax=axes[0])
sns.scatterplot(data=df.sample(300, random_state=42), x="years_since_dx", y="updrs_iii", hue="sex", ax=axes[1])
axes[0].set_title("All visits")
axes[1].set_title("300 random visits, by sex")
plt.tight_layout()
plt.show()

> Notice the cluster of points above 130? The UPDRS III scale only goes up to 132, so we'll come back to these in Section 3.

### ✅ Checkpoint 2

Before moving on, check that you can:
- summarise a numeric column with `describe()` and with individual methods (`.mean()`, `.median()`, `.std()`, `.skew()`, …);
- plot a histogram and explain how the number of bins and a log scale change what you see;
- explain the ordering of the mean, median and mode in a skewed distribution;
- compare groups with box, violin and strip plots, and say what each one shows that the others don't;
- plot two continuous variables against each other, coloured by group.

---
## 3. Outliers and data-entry errors

Outliers are values that look very different from the rest. In EHR data they have two very different causes:
- **errors**: impossible values (a negative age), unit mix-ups (g instead of mg) or typos (an extra zero);
- **genuine extremes**: rare but real patients, such as someone on a very high dose.

Statistical rules can only **flag** unusual values; deciding which kind they are takes clinical judgement. The workflow is **flag → decide → document**, and you always keep an untouched raw copy of the data.

Deal with outliers *before* imputing missing data. Otherwise the errors distort the averages and models used to fill the gaps.

In [ ]:
df.describe().T   # .T (transpose) gives one row per variable, which is easier to read

Look at the `min` and `max` columns. Which values are clinically impossible?

### Spotting outliers visually

BMI should form a roughly bell-shaped curve between about 18 and 35. A log-scaled y-axis makes rare values visible:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["BMI"].plot.hist(ax=axes[0], bins=30, edgecolor="black", logy=True, title="BMI (log-scaled y-axis)")
sns.boxplot(data=df, x="BMI", ax=axes[1])
axes[1].set_title("BMI box plot")
plt.tight_layout()
plt.show()

The isolated values far to the right (BMI ≈ 200–400, a misplaced decimal point) and the values below zero are almost certainly data-entry errors.

### Flagging outliers with rules

| Rule | Flags a value when… | Suits |
| --- | --- | --- |
| **IQR rule** | it lies outside Q1 − 1.5 × IQR or Q3 + 1.5 × IQR (the box-plot whiskers) | any shape, including skewed data |
| **Z-score** | it lies more than 3 SD from the mean, \|z\| > 3, where z = (x − mean) / SD | roughly normal data |

Both rules produce a **boolean mask**, which you already know how to use for filtering.

In [ ]:
bmi = df["BMI"]

# IQR rule
q1, q3 = bmi.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_mask = bmi.notna() & ~bmi.between(lower, upper)   # ~ means NOT; notna() stops missing values counting as outliers
print(f"IQR rule:     {iqr_mask.sum()} outliers outside {lower:.1f} – {upper:.1f}")

# Z-score rule
z = (bmi - bmi.mean()) / bmi.std()   # how many SDs each value lies from the mean
z_mask = z.abs() > 3
print(f"Z-score rule: {z_mask.sum()} outliers with |z| > 3 (mean {bmi.mean():.1f}, SD {bmi.std():.1f})")

The z-score rule flags far fewer values. The errors themselves inflate the mean and SD, which widens the "normal" range so much that most remaining values pass. Quartiles barely move, so the IQR rule is not fooled in the same way.

> 💡 For skewed variables, a **robust z-score** based on the median and MAD (median absolute deviation) is safer than the classic z-score.

#### <span style="color:red"> 🍎 Exercise 4: your turn! </span>

<span style="color:red">

* Which other variable in `df.describe()` looks like it has data-entry errors?
* Visualise its distribution with a histogram and a box plot.
* Count its outliers with both the IQR and the z-score rule. Do the two rules agree?

</span>

In [ ]:
# Write your code here:

### Skewed variables: LEDD

For LEDD the long right tail is *expected*, because doses escalate as Parkinson's progresses. The IQR rule would flag many valid high doses, so instead we can:
1. **flag on the log scale**, where LEDD is roughly normal; or
2. **flag only the most extreme few percent**, for example values outside the 1st–99th percentiles.

In [ ]:
log_ledd = np.log1p(df["LEDD"])   # log(LEDD + 1)
z_log = (log_ledd - log_ledd.mean()) / log_ledd.std()
print(f"|z| > 3 on the log scale: {(z_log.abs() > 3).sum()} flagged")

low, high = df["LEDD"].quantile([0.01, 0.99])   # 1st and 99th percentiles
pct_mask = df["LEDD"].notna() & ~df["LEDD"].between(low, high)
print(f"Outside 1st–99th percentile ({low:.0f}–{high:.0f} mg/day): {pct_mask.sum()} flagged")

### Capping and winsorisation

Instead of deleting extreme values, we can limit their influence:
- **Capping** replaces values beyond a fixed, domain-defined limit with that limit, e.g. a scale's maximum.
- **Winsorisation** caps at chosen *percentiles* instead: every value above the 99th percentile becomes the 99th-percentile value, and likewise at the bottom.

Both take one line with `.clip()`. Because they change recorded values, use them to make summaries robust and plots readable, never to rescue a result, and report them whenever you do.

In [ ]:
ledd_winsor = df["LEDD"].clip(lower=low, upper=high)   # low, high = 1st and 99th percentiles from above

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
df["LEDD"].plot.hist(ax=axes[0], bins=40, edgecolor="black", title="Original LEDD")
ledd_winsor.plot.hist(ax=axes[1], bins=40, edgecolor="black", title="Winsorised LEDD (1st–99th percentile)")
plt.tight_layout()
plt.show()

#### <span style="color:red"> 🍎 Exercise 5: your turn! </span>

<span style="color:red">

UPDRS III scores range from 0 to 132, so anything outside that range must be an error.

* Count how many values fall outside 0–132.
* **Cap** UPDRS III to 0–132 with `.clip()`.
* **Winsorise** UPDRS III at percentiles. Try 1%/99% and 0.5%/99.5%, and pick the level that best matches clinical expectations.
* Plot the original, capped and winsorised distributions side by side.

</span>

In [ ]:
# Write your code here:

#### <span style="color:red">Which method, capping or winsorisation, is better for cleaning UPDRS III, and why? Is either one appropriate for values that are impossible?</span>

Write your answer here:

### Applying the cleaning rules

A value outside a scale's range, or a physiologically impossible one, is an **error**, not an extreme. Capping a UPDRS III of 180 at 132 would invent a score nobody recorded. The safer choice is to set errors to `NaN` and handle them as missing data in Section 5.

From here on we work on `df_clean` and keep `df` as the untouched raw copy. Writing each rule in code means the cleaning is documented and reproducible.

In [ ]:
df_clean = df.copy()

plausible = {            # anything outside these ranges becomes NaN
    "age": (18, 110),    # adult cohort
    "BMI": (10, 70),
    "updrs_iii": (0, 132),
}
for col, (lo, hi) in plausible.items():   # each column name with its (low, high) limits
    bad = df_clean[col].notna() & ~df_clean[col].between(lo, hi)   # True where a value exists but is out of range
    df_clean.loc[bad, col] = np.nan   # .loc[rows, column] selects the cells to overwrite
    print(f"{col:>9}: {bad.sum():3d} values outside {lo}–{hi} set to NaN")

### ✅ Checkpoint 3

Before moving on, check that you can:
- tell a data-entry error from a genuine extreme, and explain why the difference matters;
- flag outliers with the IQR rule and z-scores using boolean masks, and say when each one works;
- handle a skewed variable with a log scale or percentiles;
- explain the difference between capping, winsorising and setting a value to `NaN`, and when to use each.

Outliers should trigger curiosity, not automatic deletion: visualise them, decide, and document what you did.

---
## 4. Duplicates

In longitudinal data, "duplicate" can mean three different things:

| Type | What it looks like | What to do |
| --- | --- | --- |
| **Exact duplicate** | every column matches another row (a copy-paste or merge artefact) | drop it |
| **Key duplicate** | same patient *and* same visit time, but slightly different values | inspect it, then apply a documented rule |
| **Repeat visit** | same patient at a *different* time | keep it: these are your repeated measures |

In [ ]:
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()   # removes rows identical in every column, keeping the first copy
print(f"Removed {n_before - len(df_clean)} exact duplicate rows")

For key duplicates we define the **visit key**, the columns that should identify a single visit. `keep=False` marks *every* copy, not just the second one, so we can see them side by side:

In [ ]:
visit_key = ["patient_id", "years_since_dx"]
key_dups = df_clean[df_clean.duplicated(subset=visit_key, keep=False)].sort_values(visit_key)
print(f"{len(key_dups)} rows share a visit key with another row")
key_dups.head(6)

The values drift slightly between the copies, which is typical of the same visit being entered twice. Two common rules resolve this:
- **keep the first entry**: simple, and keeps the original recorded values;
- **aggregate each visit**, for example by taking the median of each numeric column.

In [ ]:
# Option A: keep the first entry per visit
df_first = df_clean.drop_duplicates(subset=visit_key, keep="first")

# Option B: one row per visit, numeric columns summarised by their median
df_median = df_clean.groupby(visit_key, as_index=False).agg(   # as_index=False keeps the key columns as ordinary columns
    sex=("sex", "first"), ethnicity=("ethnicity", "first"),   # new_column=(source_column, how to combine)
    age=("age", "median"), BMI=("BMI", "median"),
    LEDD=("LEDD", "median"), updrs_iii=("updrs_iii", "median"),
)
print(f"Option A: {len(df_first)} rows | Option B: {len(df_median)} rows")

df_clean = df_first      # our documented choice: keep the first entry
assert not df_clean.duplicated(subset=visit_key).any()   # raises an error if any visit key still appears twice

> ⚠️ Always interpret duplicates in the context of the data source. In medication records, two identical rows on the same day may be a genuine bulk prescription rather than an error.

#### <span style="color:red"> 🍎 Exercise 6: your turn! </span>

<span style="color:red">

After cleaning, how many patients have more than one visit, and what is the largest number of visits for a single patient? *Hint: `groupby("patient_id").size()`.*

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 4

Before moving on, check that you can:
- explain the difference between exact duplicates, key duplicates and repeat visits;
- use `duplicated(subset=..., keep=False)` to inspect duplicates before removing them;
- resolve key duplicates with `drop_duplicates` or `groupby().agg`, and say why you chose that rule.

---
## 5. Missing data

### How much is missing?

Before counting, make sure every missing value really is `NaN`. Our `ethnicity` column also stores some gaps as empty strings, which `isna()` does not detect.

In [ ]:
df_clean["ethnicity"] = df_clean["ethnicity"].replace("", np.nan)

display(pd.DataFrame({
    "n_missing": df_clean.isna().sum(),
    "pct_missing": (df_clean.isna().mean() * 100).round(1),
}))

plt.figure(figsize=(8, 4))
plt.imshow(df_clean.isna(), aspect="auto", interpolation="none", cmap="gray_r")   # draws the True/False missingness table as an image
plt.xticks(range(df_clean.shape[1]), df_clean.columns, rotation=45, ha="right")   # label each column position with its name
plt.ylabel("Row")
plt.title("Missingness map (dark = missing)")
plt.tight_layout()
plt.show()

### Why is it missing?

The right way to handle missing data depends on *why* it is missing:

| Mechanism | Missingness depends on… | Example | Consequence |
| --- | --- | --- | --- |
| **MCAR**: missing completely at random | nothing | a lab analyser fails at random | complete cases are unbiased, just fewer |
| **MAR**: missing at random | other *observed* variables | UPDRS III is skipped more often at one clinic site | can be corrected by imputing from those variables |
| **MNAR**: missing not at random | the *missing value itself* | the most severe patients stop attending | cannot be fixed from the data alone |

Missingness also comes in two forms: a blank cell in an existing row (**item nonresponse**), or a visit or patient that is absent altogether (**unit nonresponse**). The second is easy to overlook because there is no row to count.

The data can never prove MAR or MNAR, but they can give evidence *against* MCAR. If rows with a missing value differ systematically from rows without, the missingness is not random. `groupby` on a missingness indicator makes that comparison in one line:

In [ ]:
ledd_missing = df_clean["LEDD"].isna().rename("LEDD missing")   # True/False per row; the name labels the output tables

# group rows by whether LEDD is missing, then compare the averages
display(df_clean.groupby(ledd_missing)[["years_since_dx", "updrs_iii", "age", "BMI"]].mean().round(1))
# % of each sex with LEDD observed / missing
display(pd.crosstab(df_clean["sex"].fillna("Unknown"), ledd_missing, normalize="index").mul(100).round(1))

Visits with missing LEDD have clearly higher UPDRS III scores and longer disease duration, while sex makes little difference. So LEDD is **not MCAR**. It is plausibly MAR, as long as we use UPDRS III when imputing it.

> Is a difference real, or just chance? Formal tests (point-biserial correlation, chi-squared, Little's MCAR test) answer that; statistical testing is covered in the next notebook.

#### <span style="color:red"> 🍎 Exercise 7: your turn! </span>

<span style="color:red">

Repeat this check for missing `updrs_iii`. Does its missingness look MCAR?

</span>

In [ ]:
# Write your code here:

### Handling missing data

**Deleting rows.** `dropna()` is safe only when little is missing and the missingness is plausibly MCAR; otherwise it biases the sample. Drop only on the columns your analysis actually uses (`subset=`) to keep as much data as possible.

In [ ]:
print(f"All rows:                       {len(df_clean)}")
print(f"Complete in every column:       {len(df_clean.dropna())}")
print(f"Complete for LEDD and UPDRS III: {len(df_clean.dropna(subset=['LEDD', 'updrs_iii']))}")

**Simple imputation** fills each gap with a single value: the median for skewed numeric variables (the mean if the variable is symmetric), and the mode or an explicit "Unknown" for categories. Adding a 0/1 **missingness indicator** first keeps a record of which values were filled in.

In [ ]:
df_simple = df_clean.copy()
df_simple["LEDD_was_missing"] = df_simple["LEDD"].isna().astype(int)   # 1 = imputed, 0 = observed

num_cols = ["age", "BMI", "LEDD", "updrs_iii"]
df_simple[num_cols] = df_simple[num_cols].fillna(df_simple[num_cols].median())   # each column is filled with its own median
df_simple["sex"] = df_simple["sex"].fillna(df_simple["sex"].mode()[0])   # mode() returns a Series; [0] takes the most common value
df_simple["ethnicity"] = df_simple["ethnicity"].fillna("Unknown")

df_simple.isna().sum()

**Model-based imputation** predicts each missing value from the patient's other variables, so it keeps the relationships between variables that a single fill value destroys. `IterativeImputer` from scikit-learn follows the MICE idea (Multiple Imputation by Chained Equations): it models each incomplete column from the others and cycles through them until the estimates settle. It only accepts numbers, so categorical predictors are first turned into 0/1 columns with `pd.get_dummies`.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # must come first: IterativeImputer is still "experimental"
from sklearn.impute import IterativeImputer

num_cols = ["age", "BMI", "LEDD", "updrs_iii", "years_since_dx"]
# numeric columns plus one 0/1 column per category (dummy_na=True adds one for "missing")
X = pd.concat([df_clean[num_cols],
               pd.get_dummies(df_clean[["sex", "ethnicity"]], dummy_na=True, dtype=float)], axis=1)

imputer = IterativeImputer(max_iter=10, random_state=0)   # up to 10 rounds of predicting each column from the others
# fit_transform returns a NumPy array, so we wrap it back into a DataFrame
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

df_mice = df_clean.copy()
df_mice[num_cols] = X_imputed[num_cols]   # copy back only the numeric columns

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
# zip pairs each panel with a (title, DataFrame) tuple
for ax, (label, data) in zip(axes, [("observed only", df_clean), ("median-imputed", df_simple), ("model-imputed", df_mice)]):
    data["LEDD"].plot.hist(ax=ax, bins=40, edgecolor="black", title=f"LEDD: {label}")
plt.tight_layout()
plt.show()

Median imputation stacks every missing value onto a single point, creating an artificial spike and shrinking the variance. Model-based imputation spreads the imputed values according to each patient's other measurements. Always compare distributions before and after imputing.

> A single imputed dataset still understates uncertainty. For publishable inference, use **multiple** imputation (e.g. `IterativeImputer(sample_posterior=True)` run several times, or the `miceforest` package) and pool the results with [Rubin's rules](https://bookdown.org/mwheymans/bookmi/rubins-rules.html).

#### Choosing a method

| Situation | Approach |
| --- | --- |
| Little missing (< 5%), plausibly MCAR | drop rows or use simple imputation |
| MCAR, 5–20% missing | simple imputation plus a missingness indicator |
| MAR | model-based or multiple imputation, using the variables that predict missingness |
| MNAR | no method fixes it on its own: run a sensitivity analysis and report the limitation |
| More than ~50% missing | consider dropping the variable |

### ✅ Checkpoint 5

Before moving on, check that you can:
- quantify and map missing data, including gaps hidden as empty strings;
- explain MCAR, MAR and MNAR with a clinical example of each;
- use `groupby` on a missingness indicator to find evidence against MCAR;
- choose between `dropna`, `fillna` and model-based imputation, and check the result visually.

---
## <span style="color:red"> 🍎 Final challenge: the bloodwork dataset </span>

Now apply the whole workflow to a new dataset. `data/pd_bloodwork.csv` holds single-visit (cross-sectional) bloodwork results with missing values. The variables are:
* `patient_id`: patient identifier
* `site` (A/B/C) and `clinic_day` (Mon–Fri): where and when the visit took place
* `age`, `sex`, `ethnicity`, `education_years`: demographics
* `bmi`: body mass index (kg/m²)
* `years_since_dx`, `LEDD` (mg/day), `updrs_iii` (0–132): Parkinson's measures
* `CRP` (mg/L), `ALT` (U/L): blood test results
* `household_income` (USD): annual household income

### 1) Map the missingness
Compute the % missing per column and draw a missingness map. Check whether any patients seem to be absent altogether (unit nonresponse). *Hint: look at the variables whose valid range you know.*

> **Deliverable:** a table of missingness rates, plus 2–3 sentences on which variables look "safe" (small, MCAR-like) and which look concerning.

In [ ]:
bloodwork = pd.read_csv("data/pd_bloodwork.csv")

# Write your code here:

Answer:

### 2) Diagnose the mechanism
For each variable with notable missingness, compare rows with and without missing values: `groupby` for continuous variables (`age`, `updrs_iii`, `years_since_dx`, …) and `pd.crosstab` for categorical ones (`site`, `clinic_day`, `sex`, …).

> **Deliverable:** for each variable, whether it looks MCAR, MAR (and which observed variables predict it), or possibly MNAR.

In [ ]:
# Write your code here:

Answer:

### 3) Impute and compare
Choose an imputation method for each variable based on your diagnosis, then apply it.

> **Deliverables:**
> * your plan: one line per variable, giving the method and the predictors used;
> * histograms comparing original and imputed distributions for **CRP**, **LEDD** and **UPDRS III**;
> * the imputed dataset saved as `pd_bloodwork_imputed.csv`.

We will release the complete dataset (with no missing values) later, so you can compare your imputations against the ground truth.

In [ ]:
# Write your code here:

Imputation plan:

---
## 📖 Putting it together: a cleaning workflow for clinical data

1. **Audit the raw data**: check ranges, duplicates and missingness, and never edit the raw copy.
2. **Fix errors**: set impossible values to `NaN`, drop exact duplicates, and resolve key duplicates with a stated rule.
3. **Flag, don't delete**: IQR, z-score and percentile rules mark values for review; plausible extremes stay in.
4. **Diagnose missingness** and choose a method to match.
5. **Check sensitivity**: does your conclusion change with vs. without winsorising, or with complete cases vs. imputed data?
6. **Document** every rule, every threshold and how many values each one affected.

> Outlier detection ≠ outlier deletion. **Errors → NaN → impute. Valid extremes → keep, and use robust methods.**

### 📚 Further reading (optional)
* [Missing data chapter, Data Analysis by Mike Nguyen](https://bookdown.org/mike/data_analysis/imputation-missing-data.html)
* [Flexible Imputation of Missing Data by Stef van Buuren](https://stefvanbuuren.name/fimd/), including [sensitivity analysis for MNAR](https://stefvanbuuren.name/fimd/sec-sensitivity.html)
* [Missing data methods, The Turing Way](https://book.the-turing-way.org/project-design/missing-data/missing-data-methods/)